EXECUTE THIS ONLY IN JUPYTER LABS

In [12]:
spark.sql("USE spark_db")
print("Database ready")

Database ready


In [13]:
import os, shutil

table_path = "D:/pyspark_udemy_codespace/setup/spark-warehouse/spark_db.db/sf_fire_calls"
if os.path.exists(table_path):
    shutil.rmtree(table_path)

spark.sql("DROP TABLE IF EXISTS spark_db.sf_fire_calls")

spark.sql("""
    CREATE TABLE IF NOT EXISTS spark_db.sf_fire_calls (
        CallNumber INT,
        UnitID STRING,
        IncidentNumber INT,
        CallType STRING,
        CallDate DATE,
        WatchDate STRING,
        CallFinalDisposition STRING,
        AvailableDtTm STRING,
        Address STRING,
        City STRING,
        Zipcode INT,
        Battalion STRING,
        StationArea STRING,
        Box STRING,
        OriginalPriority STRING,
        Priority STRING,
        FinalPriority INT,
        ALSUnit BOOLEAN,
        CallTypeGroup STRING,
        NumAlarms INT,
        UnitType STRING,
        UnitSequenceInCallDispatch INT,
        FirePreventionDistrict STRING,
        SupervisorDistrict STRING,
        Neighborhood STRING,
        Location STRING,
        RowID STRING,
        Delay FLOAT
    )
""")
print("Table created OK")

Table created OK


In [13]:
spark.sql("""SHOW TABLES IN spark_db""").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
| spark_db|     diamonds|      false|
| spark_db|sf_fire_calls|      false|
+---------+-------------+-----------+



In [14]:
DATA_PATH = "D:/pyspark_udemy_codespace/data/sf-fire-calls.csv"

sf_fire_calls_df = spark.read.format("csv") \
                             .option("header", "true") \
                             .option("inferSchema", "true") \
                             .load(DATA_PATH)
sf_fire_calls_df.show(10)

+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|       AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+------------

In [15]:
sf_fire_calls_df.count()

175296

In [20]:
sf_fire_calls_df.printSchema()

root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: date (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: timestamp (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: string (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: string (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (nullable = true)
 |

In [17]:
from pyspark.sql.functions import to_timestamp, expr, to_date, col

sf_fire_calls_df = sf_fire_calls_df.withColumns({
    "AvailableDtTm": to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"),
    "Zipcode": expr("CAST(zipcode as STRING)"),
    "FinalPriority": expr("CAST(finalpriority as STRING)"),
    "CallDate": to_date(expr("to_date(CallDate, 'dd/MM/yyyy')"))
    
})
print("AvailableDtTm, Zipcode, FinalPriority, CallDate has been type casted")

AvailableDtTm, Zipcode, FinalPriority, CallDate has been type casted


In [18]:
sf_fire_calls_df.write.mode("Overwrite").saveAsTable("spark_db.sf_fire_calls")
print("Data inserted into SF_FIRE_CALLS")

[Stage 16:=============================>                            (1 + 1) / 2]

Data inserted into SF_FIRE_CALLS


In [19]:
sf_fire_calls_data = spark.sql("SELECT * FROM spark_db.sf_fire_calls")
sf_fire_calls_data.show()

+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|      AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|      UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+---